In [ ]:
from pathlib import Path
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)


In [1]:
CATEGORY = "literatura"
# API_KEY = "6j1Bbsne3O1a9JE6XWZ4pGh7FBSNr6Bn" #PABLO
API_KEY = "UkKei8tlVIiGAKr6UUsmEyX6wRR2aBf4" #valent

# RELATIONS

In [2]:
import os
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout
from mistralai import Mistral
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CSV_PATH = str(DATA / "SUBSETS_with_relations" / "ARTICLES_SUBSETS_ES" / f"{CATEGORY}_articles_es_disjoint.csv")
OUTPUT_JSON = str(DATA / "SUBSETS_with_relations" / "RELATIONS_EXTRAITES" / f"relations_extraites_{CATEGORY}_es.json")

MODEL = "mistral-small-2506"
MODEL_title = MODEL.split("/")[-1].replace("-", "_")

if not API_KEY:
    raise ValueError("La variable d'environnement MISTRAL_API_KEY n'est pas définie.")

client = Mistral(api_key=API_KEY)

MAX_WORKERS_ARTICLES = 8   # nb d'articles traités en parallèle
MAX_WORKERS_TEXTES = 5     # nb de textes traités en parallèle DANS un article

# --- Nouveaux paramètres de robustesse -------------------------------------
ARTICLE_TIMEOUT = 10       # secondes max par article (passe 1)
REQUEST_TIMEOUT = 10       # secondes max par appel API
MAX_ROUNDS = 4             # nb total de passes (1 normale + 3 rattrapages)
TIMEOUT_GROWTH = 3.0       # facteur d'augmentation du timeout à chaque passe
RETRY_SLEEP = 5            # pause entre deux passes


# ---------------------------------------------------------------------------
# Prompt système : extraction de relations pour un graphe de connaissance
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """Eres un experto en extracción de relaciones para construir grafos de conocimiento.

Tu tarea: dada una PREGUNTA, identifica la(s) RELACIÓN(ES) que permitiría(n) 
encontrar la respuesta en un texto. La relación es un predicado (normalmente un 
verbo o locución verbal) que conecta un sujeto (src) con un objeto (dst).

REGLAS IMPORTANTES:
- NO extraigas las relaciones literalmente de la pregunta. Debes REFORMULAR.
- Identifica con precisión el predicado que, en un texto relevante, conectaría 
  la entidad conocida con la respuesta buscada.
- Puedes proponer varias relaciones si la pregunta lo requiere.
- La relación debe estar en forma afirmativa (no interrogativa).

Ejemplo:
Para la pregunta: "¿Qué hallazgo arqueológico realizó Selene Velázquez en el 
Templo de Nuestra Señora de los Dolores de Monterrey?"
La relación sería: "realizó el hallazgo de"
Esto permite construir, en un texto relevante, el triple:
(src="Selene Velázquez", relation="'realizó el hallazgo de ", dst="más de 2,200 ollas de barro")

Responde ÚNICAMENTE con un objeto JSON con esta estructura exacta:
{"relations": ["relacion1", "relacion2", ...]}
No añadas texto adicional, explicaciones ni comentarios."""


# ---------------------------------------------------------------------------
# Annulation coopérative : évite que les threads "zombies" d'une passe
# abandonnée continuent à taper l'API pendant la passe suivante.
# ---------------------------------------------------------------------------
_cancel_event = threading.Event()


def _chat_complete(messages, timeout_s=REQUEST_TIMEOUT):
    """Appel Mistral avec timeout réseau si le SDK le supporte."""
    try:
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
            timeout_ms=int(timeout_s * 1000),
        )
    except TypeError:
        # SDK plus ancien : pas de paramètre timeout_ms
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
        )


# ---------------------------------------------------------------------------
# Fonctions d'extraction
# ---------------------------------------------------------------------------

def extraire_relations(texte: str, max_retries: int = 3) -> list:
    """Interroge Mistral pour extraire les relations d'un texte."""
    if not isinstance(texte, str) or not texte.strip():
        return []

    content = ""
    for attempt in range(max_retries):
        if _cancel_event.is_set():
            return []
        try:
            response = _chat_complete([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Texto : {texte}"},
            ])
            content = response.choices[0].message.content
            data = json.loads(content)
            relations = data.get("relations", [])
            if isinstance(relations, str):
                relations = [relations]
            return [r.strip() for r in relations if r and r.strip()]

        except json.JSONDecodeError:
            print(f"  [!] Réponse non-JSON (tentative {attempt+1}) : {str(content)[:200]}")
        except Exception as e:
            print(f"  [!] Erreur API (tentative {attempt+1}) : {e}")
            time.sleep(2 * (attempt + 1))  # backoff

    return []  # échec après tous les essais


def traiter_article(article_id, textes):
    """Traite en parallèle tous les textes d'un article et renvoie (id, relations uniques)."""
    n = len(textes)
    relations_par_texte = [None] * n

    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS_TEXTES, max(n, 1))) as ex:
        futures = {ex.submit(extraire_relations, t): idx for idx, t in enumerate(textes)}
        for fut in as_completed(futures):
            idx = futures[fut]
            relations_par_texte[idx] = fut.result()

    flat = [r for sub in relations_par_texte if sub for r in sub]

    seen, uniques = set(), []
    for r in flat:
        if r not in seen:
            seen.add(r)
            uniques.append(r)

    return article_id, uniques


# ---------------------------------------------------------------------------
# Exécution d'une passe avec timeout par article
# ---------------------------------------------------------------------------

def _executer_passe(taches, resultats, max_workers, timeout_article, label=""):
    """
    Lance `taches` = [(article_id, textes), ...].
    Remplit `resultats` pour ceux qui aboutissent dans le délai.
    Renvoie la liste des tâches en échec (timeout ou exception).
    """
    if not taches:
        return []

    _cancel_event.clear()
    nb = len(taches)
    en_echec = []
    executor = ThreadPoolExecutor(max_workers=max_workers)
    try:
        futures = {executor.submit(traiter_article, aid, txt): (aid, txt)
                   for aid, txt in taches}

        deadline_global = time.time() + timeout_article * (nb / max(max_workers, 1) + 2)
        done_count = 0

        for fut in as_completed(futures, timeout=max(0.1, deadline_global - time.time())) \
                if False else _as_completed_safe(futures, deadline_global):
            aid, txt = futures[fut]
            done_count += 1
            try:
                article_id, uniques = fut.result(timeout=timeout_article)
                print(f"{label}[{done_count}/{nb}] article_id={article_id} -> {len(uniques)} relations")
                print(f"   => relations uniques : {uniques}\n")
                resultats[article_id] = uniques
            except FutTimeout:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ⏱️ TIMEOUT (>{timeout_article}s) -> reporté\n")
                en_echec.append((aid, txt))
            except Exception as e:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ❌ ERREUR : {e} -> reporté\n")
                en_echec.append((aid, txt))

        # Futures jamais terminées (bloquées) => reportées
        for fut, (aid, txt) in futures.items():
            if not fut.done():
                fut.cancel()
                print(f"{label}article_id={aid} ⏱️ BLOQUÉ -> reporté\n")
                en_echec.append((aid, txt))
    finally:
        # On n'attend pas les threads zombies : on leur demande d'arrêter
        _cancel_event.set()
        executor.shutdown(wait=False)

    return en_echec


def _as_completed_safe(futures, deadline_global):
    """as_completed borné par une deadline globale, sans lever d'exception."""
    pending = set(futures)
    while pending:
        restant = deadline_global - time.time()
        if restant <= 0:
            return
        try:
            for fut in as_completed(list(pending), timeout=restant):
                pending.discard(fut)
                yield fut
            return
        except FutTimeout:
            return


# ---------------------------------------------------------------------------
# Fonction principale (parallélisée + reprise des articles bloqués)
# ---------------------------------------------------------------------------

def main():
    df = pd.read_csv(CSV_PATH)

    required_cols = ["article_id", "question", "answer", "distractor1", "distractor2", "distractor3"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(
                f"Colonne '{col}' introuvable. Colonnes disponibles : {list(df.columns)}"
            )

    print(f"{len(df)} questions chargées depuis le CSV.\n")

    # Regroupement par article_id (un article = plusieurs questions)
    taches = []
    for article_id, group in df.groupby("article_id"):
        textes = []
        for row in group.itertuples(index=False):
            textes.extend([
                row.question,
                row.answer,
                row.distractor1,
                row.distractor2,
                row.distractor3,
            ])
        taches.append((article_id, textes))

    nb_articles = len(taches)
    print(f"{nb_articles} articles (groupés) à traiter.\n")

    resultats = {}          # {article_id: [relations...]}
    restantes = taches
    timeout = ARTICLE_TIMEOUT
    workers = MAX_WORKERS_ARTICLES

    for round_idx in range(1, MAX_ROUNDS + 1):
        if not restantes:
            break
        label = "" if round_idx == 1 else f"(reprise {round_idx-1}) "
        if round_idx > 1:
            print(f"\n🔁 Passe {round_idx} : {len(restantes)} article(s) à reprendre "
                  f"(timeout={timeout:.0f}s, workers={workers})\n")
            time.sleep(RETRY_SLEEP)

        restantes = _executer_passe(restantes, resultats, workers, timeout, label)

        # Passe suivante : plus de temps, moins de parallélisme
        timeout = timeout * TIMEOUT_GROWTH
        workers = max(1, workers // 2)

    # Les articles définitivement en échec -> liste vide (sortie homogène)
    for aid, _ in restantes:
        print(f"⚠️  article_id={aid} : échec définitif après {MAX_ROUNDS} passes -> []")
        resultats[aid] = []

    # Réordonner comme l'ordre des articles du CSV (facultatif, sortie plus lisible)
    resultats = {aid: resultats.get(aid, []) for aid, _ in taches}

    os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(resultats, f, ensure_ascii=False, indent=2)

    ok = sum(1 for v in resultats.values() if v)
    print(f"\n✅ Terminé. {ok}/{nb_articles} articles avec relations.")
    print(f"Résultats sauvegardés dans : {OUTPUT_JSON}")
    return resultats


if __name__ == "__main__":
    resultats = main()

1515 questions chargées depuis le CSV.

1515 articles (groupés) à traiter.

[1/1515] article_id=63 -> 10 relations
   => relations uniques : ['fue condecorado con', 'recibió la condecoración de', 'ocurrió en', 'se llevó a cabo en', 'tuvo lugar en', 'recibió una condecoración', 'fue nombrado', 'se le otorgó', 'recibió', 'le concedieron']

[2/1515] article_id=104 -> 10 relations
   => relations uniques : ['se convirtió en un himno de', 'fue utilizada por', 'se convirtió en', 'fue usada por', 'adoptó como himno', 'utilizó en la clasificación', 'consideró', 'empleó', 'se citó como', 'usó en la clasificación al']

[3/1515] article_id=562 -> 8 relations
   => relations uniques : ['fue fundada por', 'fue fundada en', 'colaboró en la formación de', 'fundó', 'pasó a llamarse', 'organizó', 'se conoció como', 'conocida como']

[4/1515] article_id=601 -> 7 relations
   => relations uniques : ['es considerado la encarnación de', 'es descrito como', 'encarnación de', 'describe como', 'encarna en', '

# ENTITIES 

In [3]:
import os
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout
from mistralai import Mistral
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CSV_PATH = str(DATA / "SUBSETS_with_relations" / "ARTICLES_SUBSETS_ES" / f"{CATEGORY}_articles_es_disjoint.csv")
OUTPUT_ENTITIES_JSON = str(DATA / "SUBSETS_with_relations" / "ENTITES_EXTRAITES" / f"entites_extraites_{CATEGORY}_es.json")

MODEL = "mistral-small-2506"
MODEL_title = MODEL.split("/")[-1].replace("-", "_")

if not API_KEY:
    raise ValueError("La variable d'environnement MISTRAL_API_KEY n'est pas définie.")

client = Mistral(api_key=API_KEY)

MAX_WORKERS = 8            # nb d'articles traités en parallèle

# --- Nouveaux paramètres de robustesse -------------------------------------
ARTICLE_TIMEOUT_E = 10     # secondes max par ligne/article (passe 1)
REQUEST_TIMEOUT_E = 10     # secondes max par appel API
MAX_ROUNDS_E = 4
TIMEOUT_GROWTH_E = 3.0
RETRY_SLEEP_E = 5


# ---------------------------------------------------------------------------
# Prompt système : extraction d'entités nommées
# ---------------------------------------------------------------------------

SYSTEM_PROMPT_ENTITIES = """Eres un experto en extracción de entidades nombradas para construir grafos de conocimiento.

Tu tarea: dado un TEXTO (respuesta correcta o distractores de una pregunta de comprensión), 
extrae las ENTIDADES NOMBRADAS relevantes. Considera como entidades: personas, lugares, 
platos, ingredientes, celebraciones, instituciones, fechas, obras, términos culturales, etc.

REGLAS IMPORTANTES:
- Extrae ÚNICAMENTE entidades concretas y relevantes para el contenido del texto.
- No incluyas pronombres, artículos ni palabras vacías.
- Si el texto es una lista, extrae cada elemento significativo por separado.
- Devuelve las entidades en minúsculas salvo nombres propios, manteniendo el formato original del nombre propio.

Responde ÚNICAMENTE con un objeto JSON con esta estructura exacta:
{"entities": ["entidad1", "entidad2", ...]}
No añadas texto adicional, explicaciones ni comentarios."""


_cancel_event_e = threading.Event()


def _chat_complete_e(messages, timeout_s=REQUEST_TIMEOUT_E):
    try:
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
            timeout_ms=int(timeout_s * 1000),
        )
    except TypeError:
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
        )


# ---------------------------------------------------------------------------
# Fonctions d'extraction
# ---------------------------------------------------------------------------

def extraire_entites(texte: str, max_retries: int = 3) -> list:
    """Interroge Mistral pour extraire les entités d'un texte."""
    if not isinstance(texte, str) or not texte.strip():
        return []

    content = ""
    for attempt in range(max_retries):
        if _cancel_event_e.is_set():
            return []
        try:
            response = _chat_complete_e([
                {"role": "system", "content": SYSTEM_PROMPT_ENTITIES},
                {"role": "user", "content": f"Texto : {texte}"},
            ])
            content = response.choices[0].message.content
            data = json.loads(content)
            entities = data.get("entities", [])
            if isinstance(entities, str):
                entities = [entities]
            return [e.strip() for e in entities if e and e.strip()]

        except json.JSONDecodeError:
            print(f"  [!] Réponse non-JSON (tentative {attempt+1}) : {str(content)[:200]}")
        except Exception as e:
            print(f"  [!] Erreur API (tentative {attempt+1}) : {e}")
            time.sleep(2 * (attempt + 1))

    return []


def traiter_article_entites(article_id, textes):
    """Traite en parallèle les textes d'un article et renvoie (id, entités uniques)."""
    entities_article = [None] * len(textes)

    with ThreadPoolExecutor(max_workers=max(1, len(textes))) as ex:
        futures = {ex.submit(extraire_entites, t): idx for idx, t in enumerate(textes)}
        for fut in as_completed(futures):
            idx = futures[fut]
            entities_article[idx] = fut.result()

    flat = [e for sub in entities_article if sub for e in sub]

    seen, uniques = set(), []
    for e in flat:
        if e not in seen:
            seen.add(e)
            uniques.append(e)

    return article_id, uniques


# ---------------------------------------------------------------------------
# Agrégation (identique à l'originale : plusieurs lignes -> même article_id)
# ---------------------------------------------------------------------------

def _agreger(resultats, article_id, uniques):
    if article_id in resultats:
        resultats[article_id].extend(uniques)
        seen, merged = set(), []
        for e in resultats[article_id]:
            if e not in seen:
                seen.add(e)
                merged.append(e)
        resultats[article_id] = merged
    else:
        resultats[article_id] = list(uniques)


# ---------------------------------------------------------------------------
# Exécution d'une passe avec timeout par article
# ---------------------------------------------------------------------------

def _as_completed_safe_e(futures, deadline_global):
    pending = set(futures)
    while pending:
        restant = deadline_global - time.time()
        if restant <= 0:
            return
        try:
            for fut in as_completed(list(pending), timeout=restant):
                pending.discard(fut)
                yield fut
            return
        except FutTimeout:
            return


def _executer_passe_entites(taches, resultats, max_workers, timeout_article, label=""):
    if not taches:
        return []

    _cancel_event_e.clear()
    nb = len(taches)
    en_echec = []
    executor = ThreadPoolExecutor(max_workers=max_workers)
    try:
        futures = {executor.submit(traiter_article_entites, aid, txt): (aid, txt)
                   for aid, txt in taches}

        deadline_global = time.time() + timeout_article * (nb / max(max_workers, 1) + 2)
        done_count = 0

        for fut in _as_completed_safe_e(futures, deadline_global):
            aid, txt = futures[fut]
            done_count += 1
            try:
                article_id, uniques = fut.result(timeout=timeout_article)
                print(f"{label}[{done_count}/{nb}] article_id={article_id} -> {len(uniques)} entités")
                print(f"   => entités uniques : {uniques}\n")
                _agreger(resultats, article_id, uniques)
            except FutTimeout:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ⏱️ TIMEOUT (>{timeout_article}s) -> reporté\n")
                en_echec.append((aid, txt))
            except Exception as e:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ❌ ERREUR : {e} -> reporté\n")
                en_echec.append((aid, txt))

        for fut, (aid, txt) in futures.items():
            if not fut.done():
                fut.cancel()
                print(f"{label}article_id={aid} ⏱️ BLOQUÉ -> reporté\n")
                en_echec.append((aid, txt))
    finally:
        _cancel_event_e.set()
        executor.shutdown(wait=False)

    return en_echec


# ---------------------------------------------------------------------------
# Fonction principale (parallélisée + reprise des articles bloqués)
# ---------------------------------------------------------------------------

def main_entities():
    df = pd.read_csv(CSV_PATH)

    required_cols = ["article_id", "answer", "distractor1", "distractor2", "distractor3", "question"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(
                f"Colonne '{col}' introuvable. Colonnes disponibles : {list(df.columns)}"
            )

    print(f"{len(df)} articles chargés depuis le CSV.\n")

    taches = []
    for row in df.itertuples(index=False):
        textes = [
            row.answer,
            row.distractor1,
            row.distractor2,
            row.distractor3,
            row.question,
        ]
        taches.append((row.article_id, textes))

    nb_taches = len(taches)
    resultats = {}  # {article_id: [entités agrégées...]}

    restantes = taches
    timeout = ARTICLE_TIMEOUT_E
    workers = MAX_WORKERS

    for round_idx in range(1, MAX_ROUNDS_E + 1):
        if not restantes:
            break
        label = "" if round_idx == 1 else f"(reprise {round_idx-1}) "
        if round_idx > 1:
            print(f"\n🔁 Passe {round_idx} : {len(restantes)} tâche(s) à reprendre "
                  f"(timeout={timeout:.0f}s, workers={workers})\n")
            time.sleep(RETRY_SLEEP_E)

        restantes = _executer_passe_entites(restantes, resultats, workers, timeout, label)

        timeout = timeout * TIMEOUT_GROWTH_E
        workers = max(1, workers // 2)

    for aid, _ in restantes:
        print(f"⚠️  article_id={aid} : échec définitif après {MAX_ROUNDS_E} passes -> []")
        if aid not in resultats:
            resultats[aid] = []

    # Ordre stable = ordre du CSV
    ordre = []
    for aid, _ in taches:
        if aid not in ordre:
            ordre.append(aid)
    resultats = {aid: resultats.get(aid, []) for aid in ordre}

    os.makedirs(os.path.dirname(OUTPUT_ENTITIES_JSON), exist_ok=True)
    with open(OUTPUT_ENTITIES_JSON, "w", encoding="utf-8") as f:
        json.dump(resultats, f, ensure_ascii=False, indent=2)

    ok = sum(1 for v in resultats.values() if v)
    print(f"\n✅ Terminé. {ok}/{len(resultats)} articles avec entités.")
    print(f"Entités sauvegardées dans : {OUTPUT_ENTITIES_JSON}")
    return resultats


if __name__ == "__main__":
    resultats_entites = main_entities()

1515 articles chargés depuis le CSV.

[1/1515] article_id=33719 -> 11 entités
   => entités uniques : ['yorkshire', 'inglaterra', 'buenos aires', 'desierto argentino', 'bari', 'italia', 'desierto de la puna', 'noroeste argentino', 'argentina', 'la cautiva', 'borges']

[2/1515] article_id=78742 -> 16 entités
   => entités uniques : ['sociedad chilena', 'labor periodística', 'perspectiva inclusiva', 'perspectiva razonada', 'revista de valparaíso', 'movimiento valparaísoño', 'cultura chilena', 'santiago', 'década de 1840', 'prensa de trincheras', 'la semana', 'prensa', 'postura liberal', 'sociedad', 'identidad chilena', 'siglo xix']

[3/1515] article_id=79073 -> 13 entités
   => entités uniques : ['barcelona', 'editorial anagrama', 'las pelucas de barcelona', 'calles de barcelona', 'madrid', 'las pelucas de madrid', 'calles de madrid', '2007', 'gerona', 'la nieve de gerona', 'calles de gerona', 'lautaro', 'la universidad desconocida']

[4/1515] article_id=109823 -> 10 entités
   => entité